# 154 — Attention Head Interaction

Attention heads don't work in isolation. Within a layer, heads may cooperate
(write in the same direction) or compete (oppose each other). Across layers,
heads may reinforce or cancel earlier computations. This module measures
these interactions.

## Why This Matters

Attention head interaction reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_interaction import (
    within_layer_interaction,
    cross_layer_alignment,
    attention_pattern_overlap,
    head_output_norms,
    head_pair_importance,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)

key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.arange(15)

## Within-Layer Head Interaction

Do heads within the same layer cooperate (aligned outputs) or compete (opposing)?

In [ ]:
result = within_layer_interaction(model, tokens)

for layer_info in result['per_layer']:
    print(f"Layer {layer_info['layer']}: "
          f"coop={layer_info['n_cooperative']} opp={layer_info['n_opposing']} "
          f"indep={layer_info['n_independent']}")
    for i in layer_info['interactions']:
        print(f"  H{i['head_a']}-H{i['head_b']}: cos={i['cosine_similarity']:+.3f} [{i['type']}]")
    print()

## Cross-Layer Head Alignment

Find head pairs in different layers with aligned output directions — potential
circuit connections or redundancy.

In [ ]:
result = cross_layer_alignment(model, tokens)

print(f"Aligned pairs: {result['n_aligned']}")
print(f"Reinforcing: {result['n_reinforcing']}, Canceling: {result['n_canceling']}\n")

for p in result['aligned_pairs'][:10]:
    print(f"{p['head_a']} <-> {p['head_b']}: cos={p['cosine_similarity']:+.4f} [{p['relationship']}]")

## Attention Pattern Overlap

Do heads attend to similar or different positions? Low overlap = heads
attend to complementary information.

In [ ]:
result = attention_pattern_overlap(model, tokens)

print(f"Most diverse layer: {result['most_diverse_layer']}\n")
for layer_info in result['per_layer']:
    print(f"Layer {layer_info['layer']}: mean_overlap={layer_info['mean_overlap']:.4f}")

## Head Output Norms

Which heads produce the largest outputs? Dominant heads drive the computation.

In [ ]:
result = head_output_norms(model, tokens)

print(f"Dominant head: {result['dominant_head']}  (norm={result['max_norm']:.4f})\n")
for h in result['per_head']:
    bar = '#' * int(h['relative_norm'] * 30)
    print(f"L{h['layer']}H{h['head']}: norm={h['output_norm']:.4f}  "
          f"relative={h['relative_norm']:.3f}  {bar}")

## Head Pair Importance

Find pairs of heads whose combined logit contribution is largest.
Cooperative pairs promote the same token; competing pairs oppose.

In [ ]:
result = head_pair_importance(model, tokens)

print(f"Target token: {result['target_token']}")
print(f"Cooperative pairs in top 10: {result['n_cooperative_top']}\n")

for p in result['top_pairs'][:5]:
    coop = 'COOP' if p['cooperative'] else 'COMP'
    print(f"{p['head_a']} + {p['head_b']}: combined={p['combined_magnitude']:.4f}  "
          f"({p['head_a_logit']:+.4f}, {p['head_b_logit']:+.4f}) [{coop}]")